# XL330 pair

Apply the marked equation, then change one gain at a time.
Click the red **Stop** button shown by each trial to end the loop and turn torque off.
Plots appear automatically. The final `await` lets Jupyter receive button clicks.

## Connect

Set the serial port, baud and motor ID(s). `PWM_LIMIT=0.80` allows up to 80% PWM output.

In [ ]:
import dxl_utils as dxl

PWM_LIMIT = 0.80  # duty fraction; 0.80 = 80%
lab = dxl.connect("/dev/ttyACM0", station="B1", baud=1_000_000,
                  leader=1, follower=2, pwm_limit=PWM_LIMIT, stop_button=True, demo=False)

## $u_F=k_p(q_L-q_F)+k_d(\dot q_L-\dot q_F)$

The follower pulls towards the leader position; the velocity term damps their relative motion. Commands are in A. Raise the position gain to reduce tracking error, then add damping and compare overshoot.

**Change:** `kp`: 0.04 → 0.08 → 0.12; `kd`: 0 → 0.006. Then `T`: 0.001 → 0.005 → 0.010.

In [ ]:
T = 0.001  # seconds; then compare 0.005 and 0.010
#TODO: kp, kd = 
with lab.run("1_unilateral", leader="off", dt=T) as run:
    while run.active:
        L, F = run.read()
        #TODO: uF =
        run.write(follower=uF)
        await run.wait_async()

## $u_L=-\beta I_F$

The follower tracks hand position using its internal position controller. Reflect its measured supply current to the leader: resistance should increase when the follower presses against an obstacle. Supply current is an effort proxy, not a calibrated contact-force measurement.

**Change:** `beta`: 0.25 → 0.5 → 1.0. Compare free motion and contact.

In [ ]:
T = 0.001
beta = 0.5
with lab.run("2_effort_feedback", leader="effort", follower="position", position_kp=150, dt=T) as run:
    while run.active:
        L, F = run.read()
        #TODO: uL = 
        run.write(leader=uL, follower_position=L.q)
        await run.wait_async()

## Impedance: $u_F=K(q_L-q_F)+b(\dot q_L-\dot q_F),\quad u_L=-u_F$

The two motors act like the ends of a spring and damper. When the real obstacle blocks the follower, their position difference grows and the leader pushes back on your hand. Both motors use current commands in A; no wall position is defined in software.

**Change:** `K`: 0.08 → 0.16 → 0.24 A/rad for a stiffer coupling; `b`: 0 → 0.006 → 0.012 A s/rad to damp relative motion. Compare free motion and contact with the real obstacle, then `T`: 0.001 → 0.005 → 0.010.

In [ ]:
T = 0.001
#TODO: K, b = 0.0, 0.006
with lab.run("3_impedance", leader="effort", follower="effort", dt=T) as run:
    while run.active:
        L, F = run.read()
        #TODO: uF = 
        #TODO: uL = 
        run.write(leader=uL, follower=uF)
        await run.wait_async()

## Disconnect

In [ ]:
lab.close()